# Import libraries

In [1]:
import gzip
import json
import requests
import io
import pandas as pd

# Dowload datasets from GH Archive

In [2]:
# Example URL for an hourly archive (January 1, 2026 at 3 PM UTC)
url = "https://data.gharchive.org/2026-01-01-12.json.gz"

In [3]:
print(f"Downloading data from {url}...")
response = requests.get(url, stream=True, timeout=30)
response.raise_for_status()

In [4]:
print(response.content[:100])  # Print the first 100 bytes of the response content

b'\x1f\x8b\x08\x08\xfcpVi\x02\x032026-01-01-12.json\x00\xec\\\xebw\xd4F\x96\xff\xbe\x7f\x85\xe2\x9c\xd9\x04\xaf\xdb-\xa9\xf5h\xf90\xcc\x00&\xc4\xbb\x01\xbc\xc4IvC\xb2vI\xaa\xeeV\xac\x96:z\xd8\xe90\x9cC\xdb\x03\x01\x02y\xce\x92\xd72@`0\t3N2Iv\x93\t\x84?'


In [5]:
# Decompress and read line by line (since it's newline-delimited JSON)
event_count = 0
push_events = 0

In [ ]:
# Decompressing in-memory
with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as f:
  for line in f:
    event_count += 1
    event = json.loads(line.decode("utf-8"))
    print(event)  # Print the entire event for inspection
    # Example: Filter for PushEvents
    if event.get("type") == "PushEvent":
      push_events += 1
      repo_name = event["repo"]["name"]
      actor = event["actor"]["login"]
      # Here you could write/stream this record to PostgreSQL, DuckDB, or Parquet files

    # Just inspect the first few events
    if event_count <= 3:
      print(
          f"\nEvent Type: {event.get('type')}, Repo: {event['repo']['name']}"
      )

print(f"\nProcessed {event_count} events total.")
print(f"Found {push_events} PushEvents in this hour.")


Event Type: PushEvent, Repo: dasepsamsul/Data-Analysis

Event Type: PushEvent, Repo: gschubert05/arb-arena

Event Type: DeleteEvent, Repo: aws-aemilia-kix/Github-PR-Commit-Integration-Test-DoNotTouch-Gith-RB7P4TGB-v2-preprod-ap-northeast-3

Processed 154722 events total.
Found 115695 PushEvents in this hour.


In [5]:
events_data = []
# max_rows = 1000  # Limit to first 1000 rows for quick inspection

with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as f:
  for i, line in enumerate(f):
    # if i >= max_rows:
    #   break
    event = json.loads(line.decode("utf-8"))

    # Flatten top-level fields and key nested fields for a clean table
    flat_event = {
        "id": event.get("id"),
        "type": event.get("type"),
        "created_at": event.get("created_at"),
        "actor": event.get("actor", {}),
        # "actor_login": event.get("actor", {}).get("login"),
        "repo": event.get("repo", {}),
        "payload": event.get("payload", {}),
        "public": event.get("public"),
    }
    events_data.append(flat_event)

# Create the DataFrame
df = pd.DataFrame(events_data)


In [6]:

# Display basic info about columns and data types
print("\n--- DataFrame Info ---")
print(df.info())




--- DataFrame Info ---
<class 'pandas.DataFrame'>
RangeIndex: 154722 entries, 0 to 154721
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   id          154722 non-null  str   
 1   type        154722 non-null  str   
 2   created_at  154722 non-null  str   
 3   actor       154722 non-null  object
 4   repo        154722 non-null  object
 5   payload     154722 non-null  object
 6   public      154722 non-null  bool  
dtypes: bool(1), object(3), str(3)
memory usage: 7.2+ MB
None


In [10]:
# Display the first 5 rows
# print("\n--- First 5 Rows ---")
print(df.head())


           id               type            created_at  \
0  7118646641          PushEvent  2026-01-01T12:00:00Z   
1  7118646642          PushEvent  2026-01-01T12:00:00Z   
2  7118646643        DeleteEvent  2026-01-01T12:00:00Z   
3  5566455635  IssueCommentEvent  2026-01-01T12:00:01Z   
4  5566455640   PullRequestEvent  2026-01-01T12:00:01Z   

                                               actor  \
0  {'id': 250934060, 'login': 'dasepsamsul', 'dis...   
1  {'id': 41898282, 'login': 'github-actions[bot]...   
2  {'id': 104595665, 'login': 'aws-aemilia-kix', ...   
3  {'id': 176961590, 'login': 'gemini-code-assist...   
4  {'id': 112882, 'login': 'usta', 'display_login...   

                                                repo  \
0  {'id': 1120900126, 'name': 'dasepsamsul/Data-A...   
1  {'id': 1038951185, 'name': 'gschubert05/arb-ar...   
2  {'id': 1053834576, 'name': 'aws-aemilia-kix/Gi...   
3  {'id': 899737840, 'name': 'kyo1941/working_tim...   
4  {'id': 188828946, 'name': 'endo

In [11]:

# Display a breakdown of event types in this batch
print("\n--- Event Types Breakdown ---")
print(df["type"].value_counts())


--- Event Types Breakdown ---
type
PushEvent                        115695
PullRequestEvent                  11989
CreateEvent                        9118
IssueCommentEvent                  3619
WatchEvent                         3607
DeleteEvent                        3505
IssuesEvent                        2288
PullRequestReviewCommentEvent      1668
PullRequestReviewEvent             1533
ForkEvent                           666
ReleaseEvent                        593
MemberEvent                         188
CommitCommentEvent                   93
GollumEvent                          79
PublicEvent                          58
DiscussionEvent                      23
Name: count, dtype: int64


In [7]:
df_actor = pd.DataFrame(df["actor"].tolist())
df_repo = pd.DataFrame(df["repo"].tolist())
df_payload = pd.DataFrame(df["payload"].tolist()) 

# Actor

In [11]:
df_actor.info()

<class 'pandas.DataFrame'>
RangeIndex: 154722 entries, 0 to 154721
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   id             154722 non-null  int64
 1   login          154722 non-null  str  
 2   display_login  154722 non-null  str  
 3   gravatar_id    154722 non-null  str  
 4   url            154722 non-null  str  
 5   avatar_url     154722 non-null  str  
dtypes: int64(1), str(5)
memory usage: 7.1 MB


# Repository

In [12]:
df_repo.info()

<class 'pandas.DataFrame'>
RangeIndex: 154722 entries, 0 to 154721
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   id      154719 non-null  float64
 1   name    154719 non-null  str    
 2   url     154719 non-null  str    
dtypes: float64(1), str(2)
memory usage: 3.5 MB


In [9]:
df_payload.info()

<class 'pandas.DataFrame'>
RangeIndex: 154722 entries, 0 to 154721
Data columns (total 25 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   repository_id  115695 non-null  float64
 1   push_id        115695 non-null  float64
 2   ref            128318 non-null  str    
 3   head           115695 non-null  str    
 4   before         115695 non-null  str    
 5   ref_type       12623 non-null   str    
 6   full_ref       12623 non-null   str    
 7   pusher_type    12623 non-null   str    
 8   action         26267 non-null   str    
 9   issue          5907 non-null    object 
 10  comment        5380 non-null    object 
 11  number         11989 non-null   float64
 12  pull_request   15190 non-null   object 
 13  master_branch  9118 non-null    str    
 14  description    4139 non-null    str    
 15  label          5876 non-null    object 
 16  labels         5919 non-null    object 
 17  member         188 non-null     object 


# Payload

In [19]:
df_payload["issue"]

0                                                       NaN
1                                                       NaN
2                                                       NaN
3         {'url': 'https://api.github.com/repos/kyo1941/...
4                                                       NaN
                                ...                        
154717                                                  NaN
154718                                                  NaN
154719                                                  NaN
154720                                                  NaN
154721                                                  NaN
Name: issue, Length: 154722, dtype: object

In [10]:
df_payload_issue = pd.DataFrame(df_payload["issue"].dropna().tolist())
df_payload_pull_request = pd.DataFrame(df_payload["pull_request"].dropna().tolist())
df_payload_comment = pd.DataFrame(df_payload["comment"].dropna().tolist())
df_payload_label = pd.DataFrame(df_payload["label"].dropna().tolist())
df_payload_labels = pd.DataFrame(df_payload["labels"].dropna().tolist())
df_payload_member = pd.DataFrame(df_payload["member"].dropna().tolist())
df_payload_release = pd.DataFrame(df_payload["release"].dropna().tolist())
df_payload_forkee = pd.DataFrame(df_payload["forkee"].dropna().tolist())
df_payload_review = pd.DataFrame(df_payload["review"].dropna().tolist())
df_payload_pages = pd.DataFrame(df_payload["pages"].dropna().tolist())
df_payload_assignee = pd.DataFrame(df_payload["assignee"].dropna().tolist())
df_payload_assignees = pd.DataFrame(df_payload["assignees"].dropna().tolist())
df_payload_discussion = pd.DataFrame(df_payload["discussion"].dropna().tolist())

## Payload - Issue

In [8]:
df_payload_issue.info()

<class 'pandas.DataFrame'>
RangeIndex: 5907 entries, 0 to 5906
Data columns (total 33 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   url                         5907 non-null   str   
 1   repository_url              5907 non-null   str   
 2   labels_url                  5907 non-null   str   
 3   comments_url                5907 non-null   str   
 4   events_url                  5907 non-null   str   
 5   html_url                    5907 non-null   str   
 6   id                          5907 non-null   int64 
 7   node_id                     5907 non-null   str   
 8   number                      5907 non-null   int64 
 9   title                       5907 non-null   str   
 10  user                        5907 non-null   object
 11  labels                      5907 non-null   object
 12  state                       5907 non-null   str   
 13  locked                      5907 non-null   bool  
 14  ass

### Payload - Issue - Type

In [23]:
payload_issue_type = pd.DataFrame(df_payload_issue["type"].dropna().tolist())
payload_issue_type.info()

<class 'pandas.DataFrame'>
RangeIndex: 199 entries, 0 to 198
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           199 non-null    int64
 1   node_id      199 non-null    str  
 2   name         199 non-null    str  
 3   description  197 non-null    str  
 4   color        199 non-null    str  
 5   created_at   199 non-null    str  
 6   updated_at   199 non-null    str  
 7   is_enabled   199 non-null    bool 
dtypes: bool(1), int64(1), str(6)
memory usage: 11.2 KB


### Payload - Issue - issue_dependencies_summary

In [22]:
payload_issue_issue_dependencies_summary = pd.DataFrame(df_payload_issue["issue_dependencies_summary"].dropna().tolist())
payload_issue_issue_dependencies_summary.info()

<class 'pandas.DataFrame'>
RangeIndex: 3156 entries, 0 to 3155
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   blocked_by        3156 non-null   int64
 1   total_blocked_by  3156 non-null   int64
 2   blocking          3156 non-null   int64
 3   total_blocking    3156 non-null   int64
dtypes: int64(4)
memory usage: 98.8 KB


### Payload - Issue - sub_issues_summary 

In [21]:
payload_issue_sub_issues_summary = pd.DataFrame(df_payload_issue["sub_issues_summary"].dropna().tolist())
payload_issue_sub_issues_summary.info()

<class 'pandas.DataFrame'>
RangeIndex: 3156 entries, 0 to 3155
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   total              3156 non-null   int64
 1   completed          3156 non-null   int64
 2   percent_completed  3156 non-null   int64
dtypes: int64(3)
memory usage: 74.1 KB


### Payload - Issue -  performed_via_github_app

In [20]:
payload_issue_performed_via_github_app = pd.DataFrame(df_payload_issue["performed_via_github_app"].dropna().tolist())
payload_issue_performed_via_github_app.dropna().head()

,id,client_id,slug,node_id,owner,name,description,external_url,html_url,created_at,updated_at,permissions,events
0,1193250,Iv23liK5TQhqbZN6WJjt,stilla,A_kwHOCznXxs4AEjUi,"{'login': 'stilla', 'id': 188340166, 'node_id'...",Stilla,Stilla,https://stilla.ai/,https://github.com/apps/stilla,2025-03-26T15:41:39Z,2025-11-19T08:07:07Z,"{'checks': 'read', 'contents': 'write', 'deplo...","[check_run, check_suite, commit_comment, creat..."
1,20150,Iv1.085d682ea245c945,linear,MDM6QXBwMjAxNTA=,"{'login': 'linear', 'id': 46686594, 'node_id':...",Linear,,https://linear.app,https://github.com/apps/linear,2018-11-02T18:07:23Z,2025-12-15T10:07:17Z,"{'checks': 'read', 'issues': 'write', 'metadat...","[check_suite, issues, issue_comment, pull_requ..."
2,20150,Iv1.085d682ea245c945,linear,MDM6QXBwMjAxNTA=,"{'login': 'linear', 'id': 46686594, 'node_id':...",Linear,,https://linear.app,https://github.com/apps/linear,2018-11-02T18:07:23Z,2025-12-15T10:07:17Z,"{'checks': 'read', 'issues': 'write', 'metadat...","[check_suite, issues, issue_comment, pull_requ..."
3,20150,Iv1.085d682ea245c945,linear,MDM6QXBwMjAxNTA=,"{'login': 'linear', 'id': 46686594, 'node_id':...",Linear,,https://linear.app,https://github.com/apps/linear,2018-11-02T18:07:23Z,2025-12-15T10:07:17Z,"{'checks': 'read', 'issues': 'write', 'metadat...","[check_suite, issues, issue_comment, pull_requ..."
4,1062359,Iv23lizrrwLGm04hszl7,openhands-ai,A_kwHOChRZg84AEDXX,"{'login': 'All-Hands-AI', 'id': 169105795, 'no...",OpenHands AI,Github App for [OpenHands Cloud](https://app.a...,https://app.all-hands.dev,https://github.com/apps/openhands-ai,2024-11-18T18:39:58Z,2025-07-13T15:23:31Z,"{'actions': 'write', 'contents': 'write', 'ema...","[commit_comment, issues, issue_comment, pull_r..."


### Payload - Issue - Reactions

In [19]:
payload_issue_reactions = pd.DataFrame(df_payload_issue["reactions"].dropna().tolist())
payload_issue_reactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 5907 entries, 0 to 5906
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   url          5907 non-null   str  
 1   total_count  5907 non-null   int64
 2   +1           5907 non-null   int64
 3   -1           5907 non-null   int64
 4   laugh        5907 non-null   int64
 5   hooray       5907 non-null   int64
 6   confused     5907 non-null   int64
 7   heart        5907 non-null   int64
 8   rocket       5907 non-null   int64
 9   eyes         5907 non-null   int64
dtypes: int64(9), str(1)
memory usage: 461.6 KB


### Payload - Issue - Pull Request

In [18]:
payload_issue_pull_request = pd.DataFrame(df_payload_issue["pull_request"].dropna().tolist())
payload_issue_pull_request.info()

<class 'pandas.DataFrame'>
RangeIndex: 2751 entries, 0 to 2750
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   url        2751 non-null   str  
 1   html_url   2751 non-null   str  
 2   diff_url   2751 non-null   str  
 3   patch_url  2751 non-null   str  
 4   merged_at  459 non-null    str  
dtypes: str(5)
memory usage: 107.6 KB


## Payload - Issue - Draft

In [16]:
payload_issue_draft = pd.DataFrame(df_payload_issue["draft"].dropna().tolist())
payload_issue_draft.info()

<class 'pandas.DataFrame'>
RangeIndex: 2751 entries, 0 to 2750
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   0       2751 non-null   bool 
dtypes: bool(1)
memory usage: 2.8 KB


In [17]:
payload_issue_draft.head()

,0
0,False
1,False
2,False
3,False
4,True


### Payload - Issue - Milestone

In [13]:
payload_issue_milestone = pd.DataFrame(df_payload_issue["milestone"].dropna().tolist())
payload_issue_milestone.info()

<class 'pandas.DataFrame'>
RangeIndex: 161 entries, 0 to 160
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   url            161 non-null    str   
 1   html_url       161 non-null    str   
 2   labels_url     161 non-null    str   
 3   id             161 non-null    int64 
 4   node_id        161 non-null    str   
 5   number         161 non-null    int64 
 6   title          161 non-null    str   
 7   description    147 non-null    str   
 8   creator        161 non-null    object
 9   open_issues    161 non-null    int64 
 10  closed_issues  161 non-null    int64 
 11  state          161 non-null    str   
 12  created_at     161 non-null    str   
 13  updated_at     161 non-null    str   
 14  due_on         63 non-null     str   
 15  closed_at      3 non-null      str   
dtypes: int64(4), object(1), str(11)
memory usage: 20.3+ KB


#### Payload - Issue - Milestone - Creator

In [15]:
payload_issue_milestone_creator = pd.DataFrame(payload_issue_milestone["creator"].dropna().tolist())
payload_issue_milestone_creator.info()

<class 'pandas.DataFrame'>
RangeIndex: 161 entries, 0 to 160
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   login                161 non-null    str  
 1   id                   161 non-null    int64
 2   node_id              161 non-null    str  
 3   avatar_url           161 non-null    str  
 4   gravatar_id          161 non-null    str  
 5   url                  161 non-null    str  
 6   html_url             161 non-null    str  
 7   followers_url        161 non-null    str  
 8   following_url        161 non-null    str  
 9   gists_url            161 non-null    str  
 10  starred_url          161 non-null    str  
 11  subscriptions_url    161 non-null    str  
 12  organizations_url    161 non-null    str  
 13  repos_url            161 non-null    str  
 14  events_url           161 non-null    str  
 15  received_events_url  161 non-null    str  
 16  type                 161 non-null    

### Payload - Issue - Assignee

In [9]:
payload_issue_assignee = pd.DataFrame(df_payload_issue["assignee"].dropna().tolist())
payload_issue_assignee.info()

<class 'pandas.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   login                1014 non-null   str  
 1   id                   1014 non-null   int64
 2   node_id              1014 non-null   str  
 3   avatar_url           1014 non-null   str  
 4   gravatar_id          1014 non-null   str  
 5   url                  1014 non-null   str  
 6   html_url             1014 non-null   str  
 7   followers_url        1014 non-null   str  
 8   following_url        1014 non-null   str  
 9   gists_url            1014 non-null   str  
 10  starred_url          1014 non-null   str  
 11  subscriptions_url    1014 non-null   str  
 12  organizations_url    1014 non-null   str  
 13  repos_url            1014 non-null   str  
 14  events_url           1014 non-null   str  
 15  received_events_url  1014 non-null   str  
 16  type                 1014 non-null 

### Payload - Issue - Assignees

In [10]:
payload_issue_assignees = pd.DataFrame(df_payload_issue["assignees"].dropna().tolist())
payload_issue_assignees.info()

<class 'pandas.DataFrame'>
RangeIndex: 5907 entries, 0 to 5906
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       1218 non-null   object
 1   1       360 non-null    object
 2   2       26 non-null     object
 3   3       17 non-null     object
 4   4       6 non-null      object
dtypes: object(5)
memory usage: 230.9+ KB


In [ ]:
payload_issue_assignees.dropna().head()

,0,1,2,3,4
5283,"{'login': 'tgolen', 'id': 1228807, 'node_id': ...","{'login': 'blimpich', 'id': 17171674, 'node_id...","{'login': 'parasharrajat', 'id': 24370807, 'no...","{'login': 'shubham1206agra', 'id': 58412969, '...","{'login': 'lydiabarclay', 'id': 195961549, 'no..."
5892,"{'login': 'Minsheng', 'id': 2377617, 'node_id'...","{'login': 'donavanbecker', 'id': 9875439, 'nod...","{'login': 'chenliuyun', 'id': 13832263, 'node_...","{'login': 'SwitchBot', 'id': 25906458, 'node_i...","{'login': 'SwitchBot-Wonderlabs', 'id': 732692..."
5893,"{'login': 'Minsheng', 'id': 2377617, 'node_id'...","{'login': 'donavanbecker', 'id': 9875439, 'nod...","{'login': 'chenliuyun', 'id': 13832263, 'node_...","{'login': 'SwitchBot', 'id': 25906458, 'node_i...","{'login': 'SwitchBot-Wonderlabs', 'id': 732692..."
5897,"{'login': 'Minsheng', 'id': 2377617, 'node_id'...","{'login': 'donavanbecker', 'id': 9875439, 'nod...","{'login': 'chenliuyun', 'id': 13832263, 'node_...","{'login': 'SwitchBot', 'id': 25906458, 'node_i...","{'login': 'SwitchBot-Wonderlabs', 'id': 732692..."
5898,"{'login': 'Minsheng', 'id': 2377617, 'node_id'...","{'login': 'donavanbecker', 'id': 9875439, 'nod...","{'login': 'chenliuyun', 'id': 13832263, 'node_...","{'login': 'SwitchBot', 'id': 25906458, 'node_i...","{'login': 'SwitchBot-Wonderlabs', 'id': 732692..."


### Payload - Issue - Users

In [37]:
df_payload_issue_users = pd.DataFrame(df_payload_issue["user"].dropna().tolist())
df_payload_issue_users.info()

<class 'pandas.DataFrame'>
RangeIndex: 5907 entries, 0 to 5906
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   login                5907 non-null   str  
 1   id                   5907 non-null   int64
 2   node_id              5907 non-null   str  
 3   avatar_url           5907 non-null   str  
 4   gravatar_id          5907 non-null   str  
 5   url                  5907 non-null   str  
 6   html_url             5907 non-null   str  
 7   followers_url        5907 non-null   str  
 8   following_url        5907 non-null   str  
 9   gists_url            5907 non-null   str  
 10  starred_url          5907 non-null   str  
 11  subscriptions_url    5907 non-null   str  
 12  organizations_url    5907 non-null   str  
 13  repos_url            5907 non-null   str  
 14  events_url           5907 non-null   str  
 15  received_events_url  5907 non-null   str  
 16  type                 5907 non-null 

### Payload - Issue - Labels

In [34]:
df_payload_issue_labels = pd.DataFrame(df_payload_issue["labels"].dropna().tolist())
df_payload_issue_labels.info()

<class 'pandas.DataFrame'>
RangeIndex: 5907 entries, 0 to 5906
Data columns (total 23 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       3260 non-null   object
 1   1       1981 non-null   object
 2   2       722 non-null    object
 3   3       367 non-null    object
 4   4       187 non-null    object
 5   5       101 non-null    object
 6   6       55 non-null     object
 7   7       33 non-null     object
 8   8       13 non-null     object
 9   9       12 non-null     object
 10  10      10 non-null     object
 11  11      5 non-null      object
 12  12      5 non-null      object
 13  13      4 non-null      object
 14  14      4 non-null      object
 15  15      4 non-null      object
 16  16      2 non-null      object
 17  17      1 non-null      object
 18  18      1 non-null      object
 19  19      1 non-null      object
 20  20      1 non-null      object
 21  21      1 non-null      object
 22  22      1 non-null      object
dtyp

In [ ]:
df_payload_issue_labels.dropna().head()

,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,20,21,22
763,"{'id': 105152717, 'node_id': 'MDU6TGFiZWwxMDUx...","{'id': 116719829, 'node_id': 'MDU6TGFiZWwxMTY3...","{'id': 122775691, 'node_id': 'MDU6TGFiZWwxMjI3...","{'id': 125550211, 'node_id': 'MDU6TGFiZWwxMjU1...","{'id': 136601536, 'node_id': 'MDU6TGFiZWwxMzY2...","{'id': 138247961, 'node_id': 'MDU6TGFiZWwxMzgy...","{'id': 169428334, 'node_id': 'MDU6TGFiZWwxNjk0...","{'id': 173493665, 'node_id': 'MDU6TGFiZWwxNzM0...","{'id': 173493835, 'node_id': 'MDU6TGFiZWwxNzM0...","{'id': 173494222, 'node_id': 'MDU6TGFiZWwxNzM0...",...,"{'id': 404091735, 'node_id': 'MDU6TGFiZWw0MDQw...","{'id': 450823910, 'node_id': 'MDU6TGFiZWw0NTA4...","{'id': 477397086, 'node_id': 'MDU6TGFiZWw0Nzcz...","{'id': 483069764, 'node_id': 'MDU6TGFiZWw0ODMw...","{'id': 687565804, 'node_id': 'MDU6TGFiZWw2ODc1...","{'id': 732478758, 'node_id': 'MDU6TGFiZWw3MzI0...","{'id': 1111992057, 'node_id': 'MDU6TGFiZWwxMTE...","{'id': 1391972461, 'node_id': 'MDU6TGFiZWwxMzk...","{'id': 2389815605, 'node_id': 'MDU6TGFiZWwyMzg...","{'id': 7581107240, 'node_id': 'LA_kwDOAToIks8A..."


In [32]:
df_labels_exploded = df_payload_issue["labels"].dropna().explode().reset_index(drop=True)

# 2. Convert that series of dictionaries into a clean DataFrame
df_labels_normalized = pd.DataFrame(df_labels_exploded.tolist())

# 3. Check the info now
df_labels_normalized.info()
df_labels_normalized.head()

<class 'pandas.DataFrame'>
RangeIndex: 9418 entries, 0 to 9417
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       6771 non-null   object
dtypes: object(1)
memory usage: 73.7+ KB


,0
0,NaN
1,NaN
2,NaN
3,NaN
4,"{'id': 2242146870, 'node_id': 'MDU6TGFiZWwyMjQ..."


In [33]:
# df_labels_normalized = df_labels_normalized[0].dropna().explode().reset_index(drop=True)
df_labels_normalized.dropna().head()


# {'id': 2242146870, 'node_id': 'MDU6TGFiZWwyMjQyMTQ2ODcw', 'url': 'https://api.github.com/repos/OldUnreal/UnrealTournamentPatches/labels/rendering', 'name': 'rendering', 'color': '51b22a', 'default': False, 'description': 'This issue affects 3d rendering'}

,0
4,"{'id': 2242146870, 'node_id': 'MDU6TGFiZWwyMjQ..."
5,"{'id': 3371654974, 'node_id': 'MDU6TGFiZWwzMzc..."
6,"{'id': 3569489676, 'node_id': 'LA_kwDOGFgg987U..."
12,"{'id': 2325523735, 'node_id': 'MDU6TGFiZWwyMzI..."
13,"{'id': 9703663217, 'node_id': 'LA_kwDOEXbUDM8A..."


## Payload - Assignee

In [22]:
df_payload_assignee.info()

<class 'pandas.DataFrame'>
RangeIndex: 555 entries, 0 to 554
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   login                555 non-null    str  
 1   id                   555 non-null    int64
 2   node_id              555 non-null    str  
 3   avatar_url           555 non-null    str  
 4   gravatar_id          555 non-null    str  
 5   url                  555 non-null    str  
 6   html_url             555 non-null    str  
 7   followers_url        555 non-null    str  
 8   following_url        555 non-null    str  
 9   gists_url            555 non-null    str  
 10  starred_url          555 non-null    str  
 11  subscriptions_url    555 non-null    str  
 12  organizations_url    555 non-null    str  
 13  repos_url            555 non-null    str  
 14  events_url           555 non-null    str  
 15  received_events_url  555 non-null    str  
 16  type                 555 non-null    

## Payload - Assignees

In [23]:
df_payload_assignees.info()

<class 'pandas.DataFrame'>
RangeIndex: 559 entries, 0 to 558
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       555 non-null    object
 1   1       270 non-null    object
 2   2       17 non-null     object
 3   3       14 non-null     object
 4   4       5 non-null      object
dtypes: object(5)
memory usage: 22.0+ KB


In [38]:
df_payload_assignees_normalized = df_payload["assignees"].dropna().explode().reset_index(drop=True)

df_payload_assignees_normalized.head()
# df_payload_assignees_normalized.info()

0    {'login': 'FrancoStino', 'id': 32127923, 'node...
1    {'login': 'karlokarate', 'id': 185203451, 'nod...
2    {'login': 'Copilot', 'id': 198982749, 'node_id...
3    {'login': 'karlokarate', 'id': 185203451, 'nod...
4    {'login': 'Copilot', 'id': 198982749, 'node_id...
Name: assignees, dtype: object

In [24]:
df_payload_comment.info()

<class 'pandas.DataFrame'>
RangeIndex: 5380 entries, 0 to 5379
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   url                       5380 non-null   str    
 1   html_url                  5380 non-null   str    
 2   issue_url                 3619 non-null   str    
 3   id                        5380 non-null   int64  
 4   node_id                   5380 non-null   str    
 5   user                      5380 non-null   object 
 6   created_at                5380 non-null   str    
 7   updated_at                5380 non-null   str    
 8   body                      5380 non-null   str    
 9   reactions                 5380 non-null   object 
 10  performed_via_github_app  2082 non-null   object 
 11  position                  1685 non-null   float64
 12  line                      0 non-null      float64
 13  path                      1685 non-null   str    
 14  commit_id          

## Payload - Discussion

In [25]:
df_payload_discussion.info()

<class 'pandas.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   repository_url      23 non-null     str   
 1   category            23 non-null     object
 2   answer_html_url     0 non-null      object
 3   answer_chosen_at    0 non-null      object
 4   answer_chosen_by    0 non-null      object
 5   html_url            23 non-null     str   
 6   id                  23 non-null     int64 
 7   node_id             23 non-null     str   
 8   number              23 non-null     int64 
 9   title               23 non-null     str   
 10  user                23 non-null     object
 11  labels              23 non-null     object
 12  state               23 non-null     str   
 13  state_reason        0 non-null      object
 14  locked              23 non-null     bool  
 15  comments            23 non-null     int64 
 16  created_at          23 non-null     str

### Payload - Discussion - reactions

In [46]:
payload_discussion_reactions = pd.DataFrame(df_payload_discussion["reactions"].dropna().tolist())
payload_discussion_reactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   url          23 non-null     str  
 1   total_count  23 non-null     int64
 2   +1           23 non-null     int64
 3   -1           23 non-null     int64
 4   laugh        23 non-null     int64
 5   hooray       23 non-null     int64
 6   confused     23 non-null     int64
 7   heart        23 non-null     int64
 8   rocket       23 non-null     int64
 9   eyes         23 non-null     int64
dtypes: int64(9), str(1)
memory usage: 1.9 KB


### Payload - Discussion - state_reason

In [45]:
payload_discussion_state_reason = pd.DataFrame(df_payload_discussion["state_reason"].dropna().tolist())
payload_discussion_state_reason.info()

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame


### Payload - Discussion - labels

In [44]:
payload_discussions_labels = pd.DataFrame(df_payload_discussion["labels"].dropna().tolist())
payload_discussions_labels.info()

<class 'pandas.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Empty DataFrame


### Payload - Discussion - User 

In [43]:
payload_discussion_user = pd.DataFrame(df_payload_discussion["user"].dropna().tolist())
payload_discussion_user.info()

<class 'pandas.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   login                23 non-null     str  
 1   id                   23 non-null     int64
 2   node_id              23 non-null     str  
 3   avatar_url           23 non-null     str  
 4   gravatar_id          23 non-null     str  
 5   url                  23 non-null     str  
 6   html_url             23 non-null     str  
 7   followers_url        23 non-null     str  
 8   following_url        23 non-null     str  
 9   gists_url            23 non-null     str  
 10  starred_url          23 non-null     str  
 11  subscriptions_url    23 non-null     str  
 12  organizations_url    23 non-null     str  
 13  repos_url            23 non-null     str  
 14  events_url           23 non-null     str  
 15  received_events_url  23 non-null     str  
 16  type                 23 non-null     st

### Payload - Discussion - answer_html_url

In [42]:
answer_html_url = df_payload_discussion["answer_html_url"].dropna().reset_index(drop=True)
answer_html_url.head()

Series([], Name: answer_html_url, dtype: object)

### Payload - Discussion - Category

In [39]:
payload_discussion_category = pd.DataFrame(df_payload_discussion["category"].dropna().tolist())
payload_discussion_category.info()

<class 'pandas.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             23 non-null     int64
 1   node_id        23 non-null     str  
 2   repository_id  23 non-null     int64
 3   emoji          23 non-null     str  
 4   name           23 non-null     str  
 5   description    23 non-null     str  
 6   created_at     23 non-null     str  
 7   updated_at     23 non-null     str  
 8   slug           23 non-null     str  
 9   is_answerable  23 non-null     bool 
dtypes: bool(1), int64(2), str(7)
memory usage: 1.8 KB


## Payload - Forkee

In [11]:
df_payload_forkee.info()

<class 'pandas.DataFrame'>
RangeIndex: 666 entries, 0 to 665
Data columns (total 79 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   id                           666 non-null    int64 
 1   node_id                      666 non-null    str   
 2   name                         666 non-null    str   
 3   full_name                    666 non-null    str   
 4   private                      666 non-null    bool  
 5   owner                        666 non-null    object
 6   html_url                     666 non-null    str   
 7   description                  543 non-null    str   
 8   fork                         666 non-null    bool  
 9   url                          666 non-null    str   
 10  forks_url                    666 non-null    str   
 11  keys_url                     666 non-null    str   
 12  collaborators_url            666 non-null    str   
 13  teams_url                    666 non-null    s

### Payload - Forkee - topics

In [50]:
payload_forkee_topics = pd.DataFrame(df_payload_forkee["topics"].dropna().tolist())
payload_forkee_topics.info()

<class 'pandas.DataFrame'>
RangeIndex: 666 entries, 0 to 665
Empty DataFrame


### Payload - Forkee - license

In [49]:
payload_forkee_license = pd.DataFrame(df_payload_forkee["license"].dropna().tolist())
payload_forkee_license.info()

<class 'pandas.DataFrame'>
RangeIndex: 429 entries, 0 to 428
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   key      429 non-null    str  
 1   name     429 non-null    str  
 2   spdx_id  429 non-null    str  
 3   url      361 non-null    str  
 4   node_id  429 non-null    str  
dtypes: str(5)
memory usage: 16.9 KB


### Payload - Forkee - mirror_url

In [48]:
payload_forkee_mirror_url = pd.DataFrame(df_payload_forkee["mirror_url"].dropna().tolist())
payload_forkee_mirror_url.info()

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame


### Payload - Forkee - Owner

In [47]:
payload_forkee_owner = pd.DataFrame(df_payload_forkee["owner"].dropna().tolist())
payload_forkee_owner.info()

<class 'pandas.DataFrame'>
RangeIndex: 666 entries, 0 to 665
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   login                666 non-null    str  
 1   id                   666 non-null    int64
 2   node_id              666 non-null    str  
 3   avatar_url           666 non-null    str  
 4   gravatar_id          666 non-null    str  
 5   url                  666 non-null    str  
 6   html_url             666 non-null    str  
 7   followers_url        666 non-null    str  
 8   following_url        666 non-null    str  
 9   gists_url            666 non-null    str  
 10  starred_url          666 non-null    str  
 11  subscriptions_url    666 non-null    str  
 12  organizations_url    666 non-null    str  
 13  repos_url            666 non-null    str  
 14  events_url           666 non-null    str  
 15  received_events_url  666 non-null    str  
 16  type                 666 non-null    

## Payload - Label

In [27]:
df_payload_label.info()

<class 'pandas.DataFrame'>
RangeIndex: 5876 entries, 0 to 5875
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           5876 non-null   int64
 1   node_id      5876 non-null   str  
 2   url          5876 non-null   str  
 3   name         5876 non-null   str  
 4   color        5876 non-null   str  
 5   default      5876 non-null   bool 
 6   description  4851 non-null   str  
dtypes: bool(1), int64(1), str(5)
memory usage: 281.3 KB


## Payload - Pages

In [28]:
df_payload_labels.info()

<class 'pandas.DataFrame'>
RangeIndex: 5919 entries, 0 to 5918
Data columns (total 40 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       5876 non-null   object
 1   1       4826 non-null   object
 2   2       627 non-null    object
 3   3       337 non-null    object
 4   4       173 non-null    object
 5   5       75 non-null     object
 6   6       32 non-null     object
 7   7       28 non-null     object
 8   8       17 non-null     object
 9   9       13 non-null     object
 10  10      6 non-null      object
 11  11      6 non-null      object
 12  12      6 non-null      object
 13  13      6 non-null      object
 14  14      6 non-null      object
 15  15      6 non-null      object
 16  16      6 non-null      object
 17  17      6 non-null      object
 18  18      6 non-null      object
 19  19      6 non-null      object
 20  20      6 non-null      object
 21  21      6 non-null      object
 22  22      6 non-null      object
 23 

## Payload - Member

In [29]:
df_payload_member.info()

<class 'pandas.DataFrame'>
RangeIndex: 188 entries, 0 to 187
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   login                188 non-null    str  
 1   id                   188 non-null    int64
 2   node_id              188 non-null    str  
 3   avatar_url           188 non-null    str  
 4   gravatar_id          188 non-null    str  
 5   url                  188 non-null    str  
 6   html_url             188 non-null    str  
 7   followers_url        188 non-null    str  
 8   following_url        188 non-null    str  
 9   gists_url            188 non-null    str  
 10  starred_url          188 non-null    str  
 11  subscriptions_url    188 non-null    str  
 12  organizations_url    188 non-null    str  
 13  repos_url            188 non-null    str  
 14  events_url           188 non-null    str  
 15  received_events_url  188 non-null    str  
 16  type                 188 non-null    

## Payload - Pages

In [30]:
df_payload_pages.info()

<class 'pandas.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Columns: 218 entries, 0 to 217
dtypes: object(218)
memory usage: 134.7+ KB


## Payload - Pull Request

In [12]:
df_payload_pull_request.info()

<class 'pandas.DataFrame'>
RangeIndex: 15190 entries, 0 to 15189
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   url     15190 non-null  str   
 1   id      15190 non-null  int64 
 2   number  15190 non-null  int64 
 3   head    15190 non-null  object
 4   base    15190 non-null  object
dtypes: int64(2), object(2), str(1)
memory usage: 593.5+ KB


### Payload - Pull Request - base

In [18]:
df_payload_pull_request_base = pd.DataFrame(df_payload_pull_request["base"].to_list())
df_payload_pull_request_base.info()

<class 'pandas.DataFrame'>
RangeIndex: 15190 entries, 0 to 15189
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ref     15190 non-null  str   
 1   sha     15190 non-null  str   
 2   repo    15190 non-null  object
dtypes: object(1), str(2)
memory usage: 356.1+ KB


#### Payload - Pull Request - base - repo

In [19]:
df_payload_pull_request_base_repo = pd.DataFrame(df_payload_pull_request_base["repo"].to_list())
df_payload_pull_request_base_repo.info()

<class 'pandas.DataFrame'>
RangeIndex: 15190 entries, 0 to 15189
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      15190 non-null  int64
 1   url     15190 non-null  str  
 2   name    15190 non-null  str  
dtypes: int64(1), str(2)
memory usage: 356.1 KB


### Payload - Pull Request - head

In [16]:
df_payload_pull_request_head = pd.DataFrame(df_payload_pull_request["head"].to_list())
df_payload_pull_request_head.info()

<class 'pandas.DataFrame'>
RangeIndex: 15190 entries, 0 to 15189
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ref     15190 non-null  str   
 1   sha     15190 non-null  str   
 2   repo    15190 non-null  object
dtypes: object(1), str(2)
memory usage: 356.1+ KB


#### Payload - Pull request - head - repo

In [17]:
df_payload_pull_request_head_repo = pd.DataFrame(df_payload_pull_request_head["repo"].to_list())
df_payload_pull_request_head_repo.info()

<class 'pandas.DataFrame'>
RangeIndex: 15190 entries, 0 to 15189
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      15190 non-null  int64
 1   url     15190 non-null  str  
 2   name    15190 non-null  str  
dtypes: int64(1), str(2)
memory usage: 356.1 KB


## Payload - release

In [32]:
df_payload_release.info()

<class 'pandas.DataFrame'>
RangeIndex: 593 entries, 0 to 592
Data columns (total 25 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   url                                  593 non-null    str    
 1   assets_url                           593 non-null    str    
 2   upload_url                           593 non-null    str    
 3   html_url                             593 non-null    str    
 4   id                                   593 non-null    int64  
 5   author                               593 non-null    object 
 6   node_id                              593 non-null    str    
 7   tag_name                             593 non-null    str    
 8   target_commitish                     593 non-null    str    
 9   name                                 590 non-null    str    
 10  draft                                593 non-null    bool   
 11  immutable                            593 no

### Payload - release- author

In [22]:
df_payload_release_author = pd.DataFrame(df_payload_release["author"].to_list())
df_payload_release_author.info()

<class 'pandas.DataFrame'>
RangeIndex: 593 entries, 0 to 592
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   login                593 non-null    str  
 1   id                   593 non-null    int64
 2   node_id              593 non-null    str  
 3   avatar_url           593 non-null    str  
 4   gravatar_id          593 non-null    str  
 5   url                  593 non-null    str  
 6   html_url             593 non-null    str  
 7   followers_url        593 non-null    str  
 8   following_url        593 non-null    str  
 9   gists_url            593 non-null    str  
 10  starred_url          593 non-null    str  
 11  subscriptions_url    593 non-null    str  
 12  organizations_url    593 non-null    str  
 13  repos_url            593 non-null    str  
 14  events_url           593 non-null    str  
 15  received_events_url  593 non-null    str  
 16  type                 593 non-null    

### Payload - release - assets

In [32]:
df_payload_release_assets_normalized = df_payload_release["assets"].dropna().explode().reset_index(drop=True)
df_payload_release_assets_normalized.info()

<class 'pandas.Series'>
RangeIndex: 1587 entries, 0 to 1586
Series name: assets
Non-Null Count  Dtype 
--------------  ----- 
1376 non-null   object
dtypes: object(1)
memory usage: 12.5+ KB


In [35]:
df_payload_release_assets = pd.DataFrame(df_payload_release_assets_normalized.dropna().to_list())
df_payload_release_assets.info()

<class 'pandas.DataFrame'>
RangeIndex: 1376 entries, 0 to 1375
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   url                   1376 non-null   str   
 1   id                    1376 non-null   int64 
 2   node_id               1376 non-null   str   
 3   name                  1376 non-null   str   
 4   label                 1266 non-null   str   
 5   uploader              1376 non-null   object
 6   content_type          1376 non-null   str   
 7   state                 1376 non-null   str   
 8   size                  1376 non-null   int64 
 9   digest                1376 non-null   str   
 10  download_count        1376 non-null   int64 
 11  created_at            1376 non-null   str   
 12  updated_at            1376 non-null   str   
 13  browser_download_url  1376 non-null   str   
dtypes: int64(3), object(1), str(10)
memory usage: 150.6+ KB


### Payload - release - mentions

In [36]:
df_payload_release_mentions_normalized = df_payload_release["mentions"].dropna().explode().reset_index(drop=True)
df_payload_release_mentions = pd.DataFrame(df_payload_release_mentions_normalized.to_list())
df_payload_release_mentions.info()

<class 'pandas.DataFrame'>
RangeIndex: 78 entries, 0 to 77
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   avatar_url         78 non-null     str  
 1   login              78 non-null     str  
 2   profile_name       59 non-null     str  
 3   profile_url        78 non-null     str  
 4   avatar_user_actor  78 non-null     bool 
dtypes: bool(1), str(4)
memory usage: 2.6 KB


### Payload - release - reactions

In [42]:
df_payload_release_reactions_normalized = df_payload_release["reactions"].dropna().reset_index(drop=True)
df_payload_release_reactions = pd.DataFrame(df_payload_release_reactions_normalized.to_list())
df_payload_release_reactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   url          1 non-null      str  
 1   total_count  1 non-null      int64
 2   +1           1 non-null      int64
 3   -1           1 non-null      int64
 4   laugh        1 non-null      int64
 5   hooray       1 non-null      int64
 6   confused     1 non-null      int64
 7   heart        1 non-null      int64
 8   rocket       1 non-null      int64
 9   eyes         1 non-null      int64
dtypes: int64(9), str(1)
memory usage: 212.0 bytes


## Payload - review

In [33]:
df_payload_review.info()

<class 'pandas.DataFrame'>
RangeIndex: 1533 entries, 0 to 1532
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                1533 non-null   int64 
 1   node_id           1533 non-null   str   
 2   user              1533 non-null   object
 3   body              832 non-null    str   
 4   commit_id         1533 non-null   str   
 5   submitted_at      1533 non-null   str   
 6   state             1533 non-null   str   
 7   html_url          1533 non-null   str   
 8   pull_request_url  1533 non-null   str   
 9   _links            1533 non-null   object
 10  updated_at        1533 non-null   str   
dtypes: int64(1), object(2), str(8)
memory usage: 131.9+ KB


### Payload - review - user

In [27]:
df_payload_review_user = pd.DataFrame(df_payload_review["user"].to_list())
df_payload_review_user.info()

<class 'pandas.DataFrame'>
RangeIndex: 1533 entries, 0 to 1532
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   login                1533 non-null   str  
 1   id                   1533 non-null   int64
 2   node_id              1533 non-null   str  
 3   avatar_url           1533 non-null   str  
 4   gravatar_id          1533 non-null   str  
 5   url                  1533 non-null   str  
 6   html_url             1533 non-null   str  
 7   followers_url        1533 non-null   str  
 8   following_url        1533 non-null   str  
 9   gists_url            1533 non-null   str  
 10  starred_url          1533 non-null   str  
 11  subscriptions_url    1533 non-null   str  
 12  organizations_url    1533 non-null   str  
 13  repos_url            1533 non-null   str  
 14  events_url           1533 non-null   str  
 15  received_events_url  1533 non-null   str  
 16  type                 1533 non-null 

### Payload - review - _links

In [28]:
df_payload_review_links = pd.DataFrame(df_payload_review["_links"].to_list())
df_payload_review_links.info()

<class 'pandas.DataFrame'>
RangeIndex: 1533 entries, 0 to 1532
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   html          1533 non-null   object
 1   pull_request  1533 non-null   object
dtypes: object(2)
memory usage: 24.1+ KB


#### Payload - review - _links - html

In [29]:
df_payload_review_links_html = pd.DataFrame(df_payload_review_links["html"].to_list())
df_payload_review_links_html.info()

<class 'pandas.DataFrame'>
RangeIndex: 1533 entries, 0 to 1532
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   href    1533 non-null   str  
dtypes: str(1)
memory usage: 12.1 KB


#### Payload - review - _links - pull_request

In [30]:
df_payload_review_links_pull_request = pd.DataFrame(df_payload_review_links["pull_request"].to_list())
df_payload_review_links_pull_request.info()

<class 'pandas.DataFrame'>
RangeIndex: 1533 entries, 0 to 1532
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   href    1533 non-null   str  
dtypes: str(1)
memory usage: 12.1 KB
